# Question 3 (Extension): Did Outside Groups Offset Brown's Campaign Spending Advantage?
**Ohio 2024 U.S. Senate race: Sherrod Brown (Democrat) vs Bernie Moreno (Republican). FEC data, 2023-24 cycle.**

**Question:** Outside groups (super PACs and similar) also pay for political ads. Did their spending offset Brown's campaign spending advantage?

**What this notebook answers:** Brown's campaign spent about $71.6M more than Moreno's. Outside groups spent $68.4M more helping Moreno's side than Brown's side, which matched about 95% of that lead. Brown's side ended 2024 about $3.3M ahead, and Moreno's side was ahead for 36 days (24 Sep to 30 Oct 2024). The notebook shows how the spending lined up, not what caused votes.

## Setup and loading the data

In [1]:
# Imports and output folder
import os
import textwrap
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT = "../outputs/question_3/"
os.makedirs(OUT, exist_ok=True)
pd.options.display.float_format = "{:,.2f}".format

In [2]:
# Load the two cleaned files. Campaign = the candidates' own committees; affiliated committees are kept aside
spend = pd.read_csv("../data/processed/campaign_spending.csv", parse_dates=["date"])
outside = pd.read_csv("../data/processed/independent_expenditures.csv", parse_dates=["date"])
print(len(spend), "rows in campaign_spending.csv;", len(outside), "rows in independent_expenditures.csv")

spend = spend[spend["include"] == True].copy()
out = outside[outside["include"] == True].copy()

campaign = spend[spend["group"] == "campaign"].copy()
affiliated = spend[spend["group"] == "affiliated"].copy()

campaign["who"] = campaign["candidate"].map({"Sherrod Brown": "Brown", "Bernie Moreno": "Moreno"})
affiliated["who"] = affiliated["candidate"].map({"Sherrod Brown": "Brown", "Bernie Moreno": "Moreno"})
out["who"] = out["helps"]

print(len(campaign), "campaign payments (used)")
print(len(affiliated), "affiliated payments (NOT used in this notebook, only in the sensitivity note)")
print(len(out), "outside payments (used)")

10718 rows in campaign_spending.csv; 1680 rows in independent_expenditures.csv
4700 campaign payments (used)
1212 affiliated payments (NOT used in this notebook, only in the sensitivity note)
1422 outside payments (used)


In [3]:
# Colours, labels and the shared chart function
BROWN, MORENO = "#1f5fa8", "#c0392b"
BROWN_LIGHT, MORENO_LIGHT = "#9fbbe0", "#e8aaa3"
COLORS = {"Brown": BROWN, "Moreno": MORENO}
LIGHT = {"Brown": BROWN_LIGHT, "Moreno": MORENO_LIGHT}
LABEL = {"Brown": "Brown's side", "Moreno": "Moreno's side"}
SIDES = ["Brown", "Moreno"]

SCOPE = "Campaign committees + outside groups (independent expenditures), 2023–2024"

FIGS = {}
CHARTS = []

def money_m(x):
    """Dollars -> text like $48.4M"""
    return f"${x / 1e6:,.1f}M"

def short_name(name):
    """Hover text keeps the full group name; chart labels cut it at the first '(' so they stay readable."""
    return name.split(" (")[0]

def wrap(name, width=30):
    """Break a long label over several lines."""
    return "<br>".join(textwrap.wrap(short_name(name), width))

def finish(fig, filename, title, scope, height=620, legend=True, top=140):
    """Apply the shared chart layout, save the chart as an HTML file and show it."""
    fig.update_layout(
        title={"text": f"<b>{title}</b><br><sup>{scope}</sup>", "x": 0.02, "y": 0.97, "yanchor": "top"},
        template="plotly_white", height=height, showlegend=legend,
        legend={"orientation": "h", "x": 1, "xanchor": "right", "y": 1.02, "yanchor": "bottom"},
        margin={"t": top, "b": 100, "l": 80, "r": 40}, font={"size": 13})
    fig.write_html(OUT + filename, include_plotlyjs=True)
    FIGS[filename] = fig
    CHARTS.append(filename)
    fig.show()

In [4]:
# One row per side: campaign money, outside money and the total
side = pd.DataFrame({"campaign": campaign.groupby("who")["amount"].sum(),
                     "outside": out.groupby("who")["amount"].sum()})
side["total"] = side["campaign"] + side["outside"]
side["outside_share_pct"] = side["outside"] / side["total"] * 100
side

,campaign,outside,total,outside_share_pct
who,,,,
Brown,"97,460,584.55","109,152,023.91","206,612,608.46",52.83
Moreno,"25,820,920.56","177,517,858.66","203,338,779.22",87.30


## Metric 1. Scorecard
**What it means:** the five headline numbers that answer the question in one look: Brown's campaign lead, how much more outside money helped Moreno, what is left when you add both together, how much outside money there was, and how much of each side's money came from outside groups.

In [5]:
# Headline numbers
brown_lead = side.loc["Brown", "campaign"] - side.loc["Moreno", "campaign"]
net_outside_help = side.loc["Moreno", "outside"] - side.loc["Brown", "outside"]
combined_lead = side.loc["Brown", "total"] - side.loc["Moreno", "total"]
total_outside = side["outside"].sum()
offset_pct = net_outside_help / brown_lead * 100

ahead = "Brown's side" if combined_lead > 0 else "Moreno's side"
scorecard = pd.DataFrame({
    "measure": ["Brown's campaign lead", "Net outside help (Moreno minus Brown)", "Combined result (Brown's side minus Moreno's side)",
                "Total outside spending", "Outside share of Brown's side", "Outside share of Moreno's side",
                "Net outside help as % of Brown's campaign lead"],
    "value": [brown_lead / 1e6, net_outside_help / 1e6, combined_lead / 1e6, total_outside / 1e6,
              side.loc["Brown", "outside_share_pct"], side.loc["Moreno", "outside_share_pct"], offset_pct],
    "unit": ["$M", "$M", "$M", "$M", "%", "%", "%"],
    "value_usd": [brown_lead, net_outside_help, combined_lead, total_outside, np.nan, np.nan, np.nan],
    "plain_words": ["Brown's own campaign spent more than Moreno's",
                    "Outside groups spent more helping Moreno's side",
                    f"{ahead} is ahead once outside money is added",
                    "Spent by outside groups to help Brown or Moreno",
                    "of Brown's side's money came from outside groups",
                    "of Moreno's side's money came from outside groups",
                    "of Brown's campaign lead was matched by net outside help"],
    "shown_as": ["card"] * 6 + ["note"]})
scorecard.to_csv(OUT + "01_scorecard.csv", index=False)
scorecard

,measure,value,unit,value_usd,plain_words,shown_as
0,Brown's campaign lead,71.64,$M,"71,639,663.99",Brown's own campaign spent more than Moreno's,card
1,Net outside help (Moreno minus Brown),68.37,$M,"68,365,834.75",Outside groups spent more helping Moreno's side,card
2,Combined result (Brown's side minus Moreno's s...,3.27,$M,"3,273,829.24",Brown's side is ahead once outside money is added,card
3,Total outside spending,286.67,$M,"286,669,882.57",Spent by outside groups to help Brown or Moreno,card
4,Outside share of Brown's side,52.83,%,NaN,of Brown's side's money came from outside groups,card
5,Outside share of Moreno's side,87.30,%,NaN,of Moreno's side's money came from outside groups,card
6,Net outside help as % of Brown's campaign lead,95.43,%,NaN,of Brown's campaign lead was matched by net ou...,note


In [6]:
# Chart 1: scorecard
cards = scorecard[scorecard["shown_as"] == "card"].reset_index(drop=True)
card_colors = [BROWN, MORENO, BROWN if combined_lead > 0 else MORENO, "#555555", BROWN, MORENO]
columns_x = [(0.00, 0.29), (0.355, 0.645), (0.71, 1.00)]

fig = go.Figure()
for i, row in cards.iterrows():
    x0, x1 = columns_x[i % 3]
    y0, y1 = (0.54, 0.98) if i < 3 else (0.02, 0.46)
    fig.add_shape(type="rect", xref="paper", yref="paper", x0=x0, x1=x1, y0=y0, y1=y1,
                  line={"color": "#dddddd"}, fillcolor="#fafafa", layer="below")
    fig.add_trace(go.Indicator(
        mode="number", value=row["value"],
        number={"prefix": "$" if row["unit"] == "$M" else "", "suffix": "M" if row["unit"] == "$M" else "%",
                "valueformat": ",.1f", "font": {"color": card_colors[i], "size": 40}},
        title={"text": "<b>" + "<br>".join(textwrap.wrap(row["measure"], 26)) + "</b>", "font": {"size": 13}},
        domain={"x": [x0, x1], "y": [y0 + 0.10, y1 - 0.02]}))
    fig.add_annotation(text="<br>".join(textwrap.wrap(row["plain_words"], 32)), xref="paper", yref="paper", x=(x0 + x1) / 2, y=y0 + 0.02,
                       xanchor="center", yanchor="bottom", showarrow=False, font={"size": 12, "color": "#444444"})

finish(fig, "01_scorecard.html", "Scorecard: did outside money offset Brown's lead?", SCOPE,
       legend=False, height=620, top=90)

## Metric 2. Who the outside money helped
**What it means:** of all the money outside groups spent, how much helped each side (the `helps` column, support for the candidate or attacks on the opponent).

In [7]:
# Outside money by the side it helps
who_helped = pd.DataFrame({"who_helped": ["Helped Moreno", "Helped Brown"],
                           "side": ["Moreno", "Brown"],
                           "dollars": [side.loc["Moreno", "outside"], side.loc["Brown", "outside"]],
                           "payments": [(out["who"] == "Moreno").sum(), (out["who"] == "Brown").sum()]})
who_helped["percent"] = who_helped["dollars"] / who_helped["dollars"].sum() * 100
who_helped.to_csv(OUT + "02_who_helped.csv", index=False)
who_helped

,who_helped,side,dollars,payments,percent
0,Helped Moreno,Moreno,"177,517,858.66",946,61.92
1,Helped Brown,Brown,"109,152,023.91",476,38.08


In [8]:
# Chart 2: who the outside money helped
fig = go.Figure(go.Pie(
    labels=who_helped["who_helped"], values=who_helped["dollars"], hole=0.6, sort=False, direction="clockwise",
    marker={"colors": [MORENO, BROWN]}, textinfo="label+percent", textposition="outside", textfont={"size": 16, "color": "#222222"},
    hovertemplate="<b>%{label}</b><br>$%{value:,.0f}<br>%{percent}<extra></extra>"))
fig.add_annotation(text=f"<b>{money_m(total_outside)}</b><br>outside spending", x=0.5, y=0.5, xref="paper", yref="paper",
                   xanchor="center", yanchor="middle", showarrow=False, font={"size": 17})
finish(fig, "02_who_helped_donut.html", "Who the outside money helped", SCOPE, height=560, legend=False)

## Metric 3. Campaign plus outside spending per side
**What it means:** each side's total money = the candidate's own campaign spending + outside spending that helps that candidate.

In [9]:
# Campaign plus outside money for each side
stack = pd.DataFrame({"side": [LABEL["Brown"], LABEL["Moreno"]],
                      "campaign_usd": [side.loc["Brown", "campaign"], side.loc["Moreno", "campaign"]],
                      "outside_usd": [side.loc["Brown", "outside"], side.loc["Moreno", "outside"]],
                      "total_usd": [side.loc["Brown", "total"], side.loc["Moreno", "total"]]})
stack["campaign_m"] = stack["campaign_usd"] / 1e6
stack["outside_m"] = stack["outside_usd"] / 1e6
stack["total_m"] = stack["total_usd"] / 1e6
stack["campaign_share_pct"] = stack["campaign_usd"] / stack["total_usd"] * 100
stack["outside_share_pct"] = stack["outside_usd"] / stack["total_usd"] * 100
stack.to_csv(OUT + "03_campaign_plus_outside.csv", index=False)
stack

,side,campaign_usd,outside_usd,total_usd,campaign_m,outside_m,total_m,campaign_share_pct,outside_share_pct
0,Brown's side,"97,460,584.55","109,152,023.91","206,612,608.46",97.46,109.15,206.61,47.17,52.83
1,Moreno's side,"25,820,920.56","177,517,858.66","203,338,779.22",25.82,177.52,203.34,12.70,87.30


In [10]:
# Chart 3: campaign plus outside spending
fig = go.Figure()
fig.add_trace(go.Bar(x=stack["side"], y=stack["campaign_m"], name="Campaign spending", marker_color=[BROWN, MORENO],
                     text=[f"Campaign<br>{money_m(v * 1e6)}" for v in stack["campaign_m"]], textposition="inside",
                     textfont={"color": "white", "size": 14}, insidetextanchor="middle", customdata=stack[["campaign_usd", "campaign_share_pct"]].values.tolist(),
                     hovertemplate="<b>%{x}: campaign</b><br>$%{customdata[0]:,.0f}<br>%{customdata[1]:.1f}% of this side's money<extra></extra>"))
fig.add_trace(go.Bar(x=stack["side"], y=stack["outside_m"], name="Outside groups", marker_color=[BROWN_LIGHT, MORENO_LIGHT],
                     text=[f"Outside groups<br>{money_m(v * 1e6)}" for v in stack["outside_m"]], textposition="inside",
                     textfont={"color": "#222222", "size": 14}, insidetextanchor="middle", customdata=stack[["outside_usd", "outside_share_pct"]].values.tolist(),
                     hovertemplate="<b>%{x}: outside groups</b><br>$%{customdata[0]:,.0f}<br>%{customdata[1]:.1f}% of this side's money<extra></extra>"))
fig.add_trace(go.Scatter(x=stack["side"], y=stack["total_m"], mode="text", text=[f"<b>Total {money_m(v * 1e6)}</b>" for v in stack["total_m"]],
                         textposition="top center", textfont={"size": 16}, showlegend=False, hoverinfo="skip"))
fig.update_layout(barmode="stack", yaxis={"title": "Money spent ($ millions)", "range": [0, 240]}, xaxis={"title": ""})
finish(fig, "03_campaign_plus_outside.html", "Campaign plus outside spending, per side", SCOPE, legend=False, top=100)

## Metric 4. Brown's combined lead over time
**What it means:** a running total, day by day, of (Brown's side spending − Moreno's side spending), campaign plus outside money together. Above zero = Brown's side is ahead. Below zero = Moreno's side is ahead.

In [11]:
# Daily spending for each side and the running lead
def daily_sides(camp, outs):
    """Money spent on each day (campaign + outside) by each side, for every day of 2023-2024."""
    days = pd.date_range("2023-01-01", "2024-12-31")
    def per_day(df, who):
        return df[df["who"] == who].groupby("date")["amount"].sum().reindex(days, fill_value=0)
    brown = per_day(camp, "Brown") + per_day(outs, "Brown")
    moreno = per_day(camp, "Moreno") + per_day(outs, "Moreno")
    return brown, moreno

brown_daily, moreno_daily = daily_sides(campaign, out)
lead_line = pd.DataFrame({"date": brown_daily.index,
                          "brown_side_daily_usd": brown_daily.values,
                          "moreno_side_daily_usd": moreno_daily.values})
lead_line["combined_lead_usd"] = (lead_line["brown_side_daily_usd"] - lead_line["moreno_side_daily_usd"]).cumsum()
lead_line["combined_lead_m"] = lead_line["combined_lead_usd"] / 1e6
lead_line["who_is_ahead"] = np.where(lead_line["combined_lead_usd"] > 0, LABEL["Brown"],
                                     np.where(lead_line["combined_lead_usd"] < 0, LABEL["Moreno"], "Even"))
lead_line.to_csv(OUT + "04_lead_over_time.csv", index=False)
lead_line.tail(3)

,date,brown_side_daily_usd,moreno_side_daily_usd,combined_lead_usd,combined_lead_m,who_is_ahead
728,2024-12-29,0.00,0.00,"3,240,805.97",3.24,Brown's side
729,2024-12-30,"34,345.73",40.00,"3,275,111.70",3.28,Brown's side
730,2024-12-31,0.00,"1,282.46","3,273,829.24",3.27,Brown's side


In [12]:
# Lead on key days, and the days Moreno's side was ahead
lead_by_day = lead_line.set_index("date")["combined_lead_usd"]
lowest_day = lead_by_day.idxmin()
days_moreno_ahead = lead_by_day[lead_by_day < 0]
print("Moreno's side was ahead on", len(days_moreno_ahead), "days: from", days_moreno_ahead.index.min().date(),
      "to", days_moreno_ahead.index.max().date())

milestones = pd.DataFrame({
    "milestone": ["End of 2023", "Primary", "End of June", "End of August",
                  "First day Moreno's side ahead", "End of September", "Lowest point", "Election Day", "End of 2024"],
    "date": pd.to_datetime(["2023-12-31", "2024-03-19", "2024-06-30", "2024-08-31",
                            days_moreno_ahead.index.min(), "2024-09-30", lowest_day, "2024-11-05", "2024-12-31"])})
milestones["combined_lead_usd"] = milestones["date"].map(lead_by_day)
milestones["combined_lead_m"] = milestones["combined_lead_usd"] / 1e6
milestones["who_is_ahead"] = np.where(milestones["combined_lead_usd"] > 0, LABEL["Brown"], LABEL["Moreno"])
milestones.to_csv(OUT + "04_lead_milestones.csv", index=False)
milestones

Moreno's side was ahead on 36 days: from 2024-09-24 to 2024-10-30


,milestone,date,combined_lead_usd,combined_lead_m,who_is_ahead
0,End of 2023,2023-12-31,"5,201,720.89",5.20,Brown's side
1,Primary,2024-03-19,"17,301,857.03",17.30,Brown's side
2,End of June,2024-06-30,"32,412,034.98",32.41,Brown's side
3,End of August,2024-08-31,"29,982,337.27",29.98,Brown's side
4,First day Moreno's side ahead,2024-09-24,"-4,495,838.70",-4.50,Moreno's side
5,End of September,2024-09-30,"322,560.17",0.32,Brown's side
6,Lowest point,2024-10-15,"-10,031,282.04",-10.03,Moreno's side
7,Election Day,2024-11-05,"1,895,164.81",1.90,Brown's side
8,End of 2024,2024-12-31,"3,273,829.24",3.27,Brown's side


In [13]:
# Chart 4: combined lead over time
fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.03,
                    specs=[[{"type": "xy"}, {"type": "table"}]])

hover = [f"{who} ahead by ${abs(v):,.2f}M" if who != "Even" else "Even"
         for who, v in zip(lead_line["who_is_ahead"], lead_line["combined_lead_m"])]
fig.add_trace(go.Scatter(x=lead_line["date"], y=lead_line["combined_lead_m"], mode="lines", name="Combined lead",
                         line={"color": "#222222", "width": 2}, customdata=hover,
                         hovertemplate="%{x|%d %b %Y}<br>%{customdata}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=lead_line["date"], y=lead_line["combined_lead_m"].clip(lower=0), fill="tozeroy", mode="lines",
                         line={"width": 0}, fillcolor="rgba(31,95,168,0.25)", hoverinfo="skip", name="Brown's side ahead"), row=1, col=1)
fig.add_trace(go.Scatter(x=lead_line["date"], y=lead_line["combined_lead_m"].clip(upper=0), fill="tozeroy", mode="lines",
                         line={"width": 0}, fillcolor="rgba(192,57,43,0.35)", hoverinfo="skip", name="Moreno's side ahead"), row=1, col=1)
fig.add_hline(y=0, line={"color": "black", "width": 2}, row=1, col=1)

events = [("Primary 19 Mar", "2024-03-19", None, 67),
          ("Republican convention 15–18 Jul", "2024-07-15", "2024-07-18", 62),
          ("Democratic convention 19–22 Aug", "2024-08-19", "2024-08-22", 57),
          ("Labor Day 2 Sep", "2024-09-02", None, 52),
          ("Election Day 5 Nov", "2024-11-05", None, 47)]
for label, start, end, label_y in events:
    if end is None:
        fig.add_shape(type="line", x0=start, x1=start, y0=0, y1=1, xref="x", yref="paper", line={"color": "#777777", "dash": "dot", "width": 1.5})
    else:
        fig.add_shape(type="rect", x0=start, x1=end, y0=0, y1=1, xref="x", yref="paper", fillcolor="rgba(120,120,120,0.3)", line={"width": 0})
    fig.add_annotation(x=start, y=label_y, xref="x", yref="y", text=label, showarrow=False, xanchor="right", xshift=-4,
                       bgcolor="rgba(255,255,255,0.9)", font={"size": 11})

lead_text = [("+" if v >= 0 else "−") + f"${abs(v):,.1f}M" for v in milestones["combined_lead_m"]]
who_short = milestones["who_is_ahead"].str.replace("'s side", "")
fig.add_trace(go.Table(columnwidth=[2.3, 1.5, 1.1, 1.0],
                       header={"values": ["<b>Milestone</b>", "<b>Date</b>", "<b>Lead</b>", "<b>Ahead</b>"], "fill_color": "#eeeeee", "align": "left"},
                       cells={"values": [milestones["milestone"], milestones["date"].dt.strftime("%d %b %Y"), lead_text, who_short],
                              "align": "left", "height": 34, "font": {"size": 12}}), row=1, col=2)

fig.update_xaxes(title_text="Date", tickformat="%b %Y", row=1, col=1)
fig.update_yaxes(title_text="Brown's side minus Moreno's side ($ millions, running total)", range=[-14, 72], row=1, col=1)
finish(fig, "04_lead_over_time.html", "Brown's combined lead over time (campaign + outside money)", SCOPE,
       legend=False, height=740, top=110)

## Metric 5. Outside spending by month and side
**What it means:** how much outside money helping each side was spent in each month (not including the campaigns' own spending).

In [14]:
# Outside spending by month and the September to October share
months = pd.period_range("2023-01", "2024-12", freq="M").astype(str)
monthly = out.pivot_table(index="month", columns="who", values="amount", aggfunc="sum").reindex(months).fillna(0)
monthly.index.name = "month"
monthly = monthly.reset_index().rename(columns={"Brown": "brown_side_usd", "Moreno": "moreno_side_usd"})
monthly["brown_side_m"] = monthly["brown_side_usd"] / 1e6
monthly["moreno_side_m"] = monthly["moreno_side_usd"] / 1e6
monthly["brown_pct_of_side"] = monthly["brown_side_usd"] / monthly["brown_side_usd"].sum() * 100
monthly["moreno_pct_of_side"] = monthly["moreno_side_usd"] / monthly["moreno_side_usd"].sum() * 100
monthly["shown_in_chart"] = monthly["month"] >= "2024-02"
monthly.to_csv(OUT + "05_outside_by_month.csv", index=False)

sep_oct = monthly[monthly["month"].isin(["2024-09", "2024-10"])]
sep_oct_share = pd.DataFrame({"side": [LABEL["Brown"], LABEL["Moreno"]],
                              "sep_oct_usd": [sep_oct["brown_side_usd"].sum(), sep_oct["moreno_side_usd"].sum()],
                              "side_outside_total_usd": [monthly["brown_side_usd"].sum(), monthly["moreno_side_usd"].sum()]})
sep_oct_share["sep_oct_pct"] = sep_oct_share["sep_oct_usd"] / sep_oct_share["side_outside_total_usd"] * 100
sep_oct_share.to_csv(OUT + "05_sep_oct_share.csv", index=False)
sep_oct_share

,side,sep_oct_usd,side_outside_total_usd,sep_oct_pct
0,Brown's side,"97,628,451.70","109,152,023.91",89.44
1,Moreno's side,"153,072,512.42","177,517,858.66",86.23


In [15]:
# Chart 5: outside spending by month
shown = monthly[monthly["shown_in_chart"]]
fig = go.Figure()
for who, usd, col, pct in [("Brown", "brown_side_m", "brown_side_usd", "brown_pct_of_side"), ("Moreno", "moreno_side_m", "moreno_side_usd", "moreno_pct_of_side")]:
    fig.add_trace(go.Bar(x=shown["month"], y=shown[usd], name=LABEL[who], marker_color=COLORS[who],
                         customdata=shown[[col, pct]].values.tolist(),
                         hovertemplate="<b>" + LABEL[who] + "</b><br>%{x}<br>$%{customdata[0]:,.0f}<br>%{customdata[1]:.1f}% of this side's outside money<extra></extra>"))
fig.update_layout(barmode="group", xaxis={"title": "Month", "tickangle": -45, "type": "category"},
                  yaxis={"title": "Outside spending ($ millions)"})
finish(fig, "05_outside_by_month.html", "Outside spending by month and side", "Outside groups only (independent expenditures), Feb 2024 – Dec 2024",
       height=700)

## Metric 6. Biggest outside groups
**What it means:** which groups spent the most, and how much of each side's outside money came from just a few groups. A "group" is a committee name in the file (`committee`), used as it is.

In [16]:
# Spending by outside group; attack = payments that oppose the other candidate
out["attack_amount"] = np.where(out["tone"] == "oppose", out["amount"], 0)
groups = out.groupby(["who", "committee"]).agg(payments=("amount", "size"),
                                              total=("amount", "sum"),
                                              attack=("attack_amount", "sum")).reset_index()
groups["attack_pct"] = groups["attack"] / groups["total"] * 100
print("Number of groups per side:")
print(groups.groupby("who")["committee"].nunique())

side_total = groups.groupby("who")["total"].transform("sum")
groups["pct_of_side"] = groups["total"] / side_total * 100
top10 = groups.sort_values("total", ascending=False).groupby("who").head(10).copy()
top10 = top10.sort_values(["who", "total"], ascending=[True, False]).reset_index(drop=True)
top10["rank"] = top10.groupby("who").cumcount() + 1
top10["total_m"] = top10["total"] / 1e6
top10 = top10[["who", "rank", "committee", "payments", "total", "total_m", "pct_of_side", "attack_pct"]]
top10.to_csv(OUT + "06a_top_groups.csv", index=False)
top10

Number of groups per side:
who
Brown     63
Moreno    30
Name: committee, dtype: int64


,who,rank,committee,payments,total,total_m,pct_of_side,attack_pct
0,Brown,1,WINSENATE,68,"79,327,349.79",79.33,72.68,91.10
1,Brown,2,DUTY AND COUNTRY,10,"4,032,811.36",4.03,3.69,79.18
2,Brown,3,"BUCKEYE LEADERSHIP FUND, INC.",14,"3,768,159.49",3.77,3.45,100.00
3,Brown,4,NATIONAL ASSOCIATION OF REALTORS CONGRESSIONAL...,4,"2,394,001.58",2.39,2.19,0.00
4,Brown,5,BLACKPAC,19,"2,326,734.85",2.33,2.13,0.00
5,Brown,6,LCV VICTORY FUND,16,"2,250,411.35",2.25,2.06,30.01
6,Brown,7,LEADERSHIP FOR OHIO FUND,4,"2,229,013.12",2.23,2.04,100.00
7,Brown,8,NEW AMERICAN JOBS FUND,20,"2,196,485.30",2.20,2.01,0.00
8,Brown,9,CFFE PAC,14,"2,180,920.00",2.18,2.00,0.00
9,Brown,10,EDF ACTION VOTES,4,"1,501,725.50",1.50,1.38,100.00


In [17]:
# Chart 6a: the 10 biggest groups on each side
fig = go.Figure()
ticks, x_max = {}, {}
for who in SIDES:
    t = top10[top10["who"] == who]
    ticks[who] = [wrap(n).replace("<br>", "\u00a0\u00a0\u00a0\u00a0<br>") + "\u00a0\u00a0\u00a0\u00a0" for n in t["committee"]]
    x_max[who] = t["total_m"].max() * 1.3
    fig.add_trace(go.Bar(x=t["total_m"], y=t["rank"], orientation="h", name=LABEL[who], marker_color=COLORS[who],
                         visible=(who == "Brown"), text=[f"${v:,.1f}M" for v in t["total_m"]], textposition="outside",
                         customdata=t[["committee", "total", "pct_of_side", "payments"]].values.tolist(),
                         hovertemplate="<b>%{customdata[0]}</b><br>$%{customdata[1]:,.0f}<br>%{customdata[2]:.1f}% of this side's outside money<br>%{customdata[3]} payments<extra></extra>"))

buttons = []
for i, who in enumerate(SIDES):
    buttons.append({"label": "Helped " + who, "method": "update",
                    "args": [{"visible": [j == i for j in range(len(SIDES))]},
                             {"yaxis.ticktext": ticks[who], "xaxis.range": [0, x_max[who]]}]})
fig.update_layout(
    xaxis={"title": "Total outside spending ($ millions)", "range": [0, x_max["Brown"]]},
    yaxis={"title": "", "tickmode": "array", "tickvals": list(range(1, 11)), "ticktext": ticks["Brown"], "range": [10.8, 0.2]},
    bargap=0.35,
    updatemenus=[{"type": "buttons", "direction": "right", "x": 0.0, "y": 1.12, "xanchor": "left", "buttons": buttons}],
    margin={"l": 300})
finish(fig, "06a_top_groups.html", "The 10 biggest outside groups, by side", "Outside groups only (independent expenditures); use the buttons to switch side",
       legend=False, height=760, top=170)

In [18]:
# Top 3 groups against all other groups
conc_rows = []
for who in SIDES:
    g = groups[groups["who"] == who].sort_values("total", ascending=False)
    top3 = g.head(3)
    conc_rows.append({"who": who, "slice": "Top 3 groups", "dollars": top3["total"].sum(), "groups": "; ".join(top3["committee"].map(short_name))})
    conc_rows.append({"who": who, "slice": "All other groups", "dollars": g["total"].iloc[3:].sum(), "groups": f"{len(g) - 3} other groups"})
conc = pd.DataFrame(conc_rows)
conc["percent"] = conc["dollars"] / conc.groupby("who")["dollars"].transform("sum") * 100
conc.to_csv(OUT + "06b_concentration.csv", index=False)
conc

,who,slice,dollars,groups,percent
0,Brown,Top 3 groups,"87,128,320.64",WINSENATE; DUTY AND COUNTRY; BUCKEYE LEADERSHI...,79.82
1,Brown,All other groups,"22,023,703.27",60 other groups,20.18
2,Moreno,Top 3 groups,"141,630,973.44",SENATE LEADERSHIP FUND; DEFEND AMERICAN JOBS; ...,79.78
3,Moreno,All other groups,"35,886,885.22",27 other groups,20.22


In [19]:
# Chart 6b: how concentrated the money is
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "domain"}]], horizontal_spacing=0.08)
for col, who in enumerate(SIDES, start=1):
    c = conc[conc["who"] == who]
    fig.add_trace(go.Pie(labels=c["slice"], values=c["dollars"], hole=0.6, sort=False, direction="clockwise",
                         marker={"colors": [COLORS[who], LIGHT[who]]}, textinfo="percent", textposition="inside", insidetextorientation="horizontal",
                         textfont={"size": 17, "color": ["white", "#222222"]},
                         customdata=c["groups"].str.replace("; ", "<br>").tolist(),
                         hovertemplate="<b>%{label}</b><br>$%{value:,.0f}<br>%{percent}<br>%{customdata}<extra></extra>"), row=1, col=col)
    top_pct = c.loc[c["slice"] == "Top 3 groups", "percent"].iloc[0]
    fig.add_annotation(text=f"<b>{LABEL[who]}</b><br>Top 3 = {top_pct:.0f}%", x=[0.225, 0.775][col - 1], y=0.5, xref="paper", yref="paper",
                       xanchor="center", yanchor="middle", showarrow=False, font={"size": 16})
finish(fig, "06b_concentration_donuts.html", "A few groups supply most of the outside money", "Outside groups only (independent expenditures)", height=540, legend=False)

## Metric 7. Support vs attack
**What it means:** how much of each side's outside money *supports* its own candidate and how much *attacks* the opponent (the existing `tone` column).

In [20]:
# Support and attack for each side
tone = out.groupby(["who", "tone"]).agg(dollars=("amount", "sum"), payments=("amount", "size")).reset_index()
tone["type"] = tone["tone"].map({"oppose": "Attack", "support": "Support"})
tone["percent"] = tone["dollars"] / tone.groupby("who")["dollars"].transform("sum") * 100
tone = tone.sort_values(["who", "type"]).reset_index(drop=True)
tone = tone[["who", "type", "dollars", "percent", "payments"]]
tone.to_csv(OUT + "07_support_vs_attack.csv", index=False)
tone

,who,type,dollars,percent,payments
0,Brown,Attack,"85,641,468.06",78.46,123
1,Brown,Support,"23,510,555.85",21.54,353
2,Moreno,Attack,"108,637,222.60",61.20,300
3,Moreno,Support,"68,880,636.06",38.80,646


In [21]:
# Chart 7: support and attack
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "domain"}]], horizontal_spacing=0.08)
for col, who in enumerate(SIDES, start=1):
    t = tone[tone["who"] == who]
    fig.add_trace(go.Pie(labels=t["type"], values=t["dollars"], hole=0.65, sort=False, direction="clockwise",
                         marker={"colors": [COLORS[who], LIGHT[who]]}, textinfo="label+percent", textfont={"size": 13},
                         hovertemplate="<b>%{label}</b><br>$%{value:,.0f}<br>%{percent}<extra></extra>"), row=1, col=col)
    attack_pct = t.loc[t["type"] == "Attack", "percent"].iloc[0]
    fig.add_annotation(text=f"<b>{LABEL[who]}</b><br>{attack_pct:.0f}% attack", x=[0.225, 0.775][col - 1], y=0.5, xref="paper", yref="paper",
                       xanchor="center", yanchor="middle", showarrow=False, font={"size": 16})
finish(fig, "07_support_vs_attack.html", "Support vs attack in the outside money", "Outside groups only (independent expenditures)", height=520, legend=False)

## Metric 8. Types of outside communication
**What it means:** what the outside money was spent on, using the existing `category` column as it is. Categories that are under 2% of *both* sides' outside money are grouped into "Other (small categories)" so the chart stays readable.

In [22]:
# Spending by category; categories under 2% on both sides are grouped as Other
cat = out.pivot_table(index="category", columns="who", values="amount", aggfunc="sum").fillna(0)
cat_pct = cat / cat.sum() * 100
big = cat_pct.max(axis=1) >= 2
print("Grouped into 'Other':", list(cat.index[~big]))

cat_show = cat[big].copy()
cat_show = cat_show.loc[cat_show.sum(axis=1).sort_values(ascending=False).index]
cat_show.loc["Other (small categories)"] = cat[~big].sum()

cat_long = cat_show.reset_index().melt(id_vars="category", var_name="who", value_name="dollars")
cat_long["percent"] = cat_long["dollars"] / cat_long.groupby("who")["dollars"].transform("sum") * 100
cat_long["side"] = cat_long["who"].map(LABEL)
cat_long = cat_long[["side", "who", "category", "dollars", "percent"]]
cat_long.to_csv(OUT + "08_communication_types.csv", index=False)
cat_long

Grouped into 'Other': ['Consulting & strategy', 'Fundraising costs', 'Merchandise & signs', 'Operations & overhead', 'Other / unclear', 'Payroll & benefits', 'Reimbursements & card payments', 'Texting & phone', 'Travel, events & meals']


,side,who,category,dollars,percent
0,Brown's side,Brown,"TV, radio & other media","68,992,025.34",63.21
1,Brown's side,Brown,Digital advertising,"31,819,696.46",29.15
2,Brown's side,Brown,Direct mail & printing,"1,836,412.47",1.68
3,Brown's side,Brown,Canvassing & field,"5,800,960.50",5.31
4,Brown's side,Brown,Other (small categories),"702,929.14",0.64
5,Moreno's side,Moreno,"TV, radio & other media","133,573,569.44",75.25
6,Moreno's side,Moreno,Digital advertising,"25,929,373.16",14.61
7,Moreno's side,Moreno,Direct mail & printing,"14,476,211.91",8.15
8,Moreno's side,Moreno,Canvassing & field,"1,407,292.30",0.79
9,Moreno's side,Moreno,Other (small categories),"2,131,411.85",1.20


In [23]:
# Chart 8: types of outside communication
CAT_COLORS = {"TV, radio & other media": "#6a51a3", "Digital advertising": "#e6a817", "Direct mail & printing": "#2a9d8f",
              "Canvassing & field": "#8d6e63", "Other (small categories)": "#9e9e9e"}
fig = go.Figure()
for name in cat_show.index:
    rows = cat_long[cat_long["category"] == name]
    fig.add_trace(go.Bar(y=rows["side"], x=rows["percent"], orientation="h", name=name, marker_color=CAT_COLORS.get(name, "#bbbbbb"),
                         text=[f"{p:.0f}%" if p >= 4 else "" for p in rows["percent"]], textposition="inside",
                         textfont={"color": "white"}, customdata=rows["dollars"].tolist(),
                         hovertemplate="<b>%{y}</b><br>" + name + "<br>%{x:.1f}% of this side's outside money<br>$%{customdata:,.0f}<extra></extra>"))
fig.update_layout(barmode="stack", xaxis={"title": "Share of the side's outside money (%)", "range": [0, 100]},
                  yaxis={"title": "", "autorange": "reversed"})
finish(fig, "08_communication_types.html", "What the outside money was spent on", "Outside groups only (independent expenditures); each bar adds to 100%", height=480)

## Metric 9. Each outside group as a dot
**What it means:** every outside group is one dot: how many payments it made (left to right) and how much it spent in total (bottom to top). A group that helped both sides would get one dot per side.

In [24]:
# One row per group and side
groups["total_m"] = groups["total"] / 1e6
dots = groups[["who", "committee", "payments", "total", "total_m", "attack_pct"]].sort_values(["who", "total"], ascending=[True, False]).reset_index(drop=True)
dots.to_csv(OUT + "09_group_dots.csv", index=False)
dots.head(8)

,who,committee,payments,total,total_m,attack_pct
0,Brown,WINSENATE,68,"79,327,349.79",79.33,91.10
1,Brown,DUTY AND COUNTRY,10,"4,032,811.36",4.03,79.18
2,Brown,"BUCKEYE LEADERSHIP FUND, INC.",14,"3,768,159.49",3.77,100.00
3,Brown,NATIONAL ASSOCIATION OF REALTORS CONGRESSIONAL...,4,"2,394,001.58",2.39,0.00
4,Brown,BLACKPAC,19,"2,326,734.85",2.33,0.00
5,Brown,LCV VICTORY FUND,16,"2,250,411.35",2.25,30.01
6,Brown,LEADERSHIP FOR OHIO FUND,4,"2,229,013.12",2.23,100.00
7,Brown,NEW AMERICAN JOBS FUND,20,"2,196,485.30",2.20,0.00


In [25]:
# Chart 9: each group as a dot
label_rows = dots.sort_values("total", ascending=False).head(6).index
fig = go.Figure()
for who in SIDES:
    d = dots[dots["who"] == who]
    fig.add_trace(go.Scatter(x=d["payments"], y=d["total_m"], mode="markers+text", name=LABEL[who],
                             marker={"color": COLORS[who], "size": 13, "opacity": 0.7, "line": {"color": "white", "width": 1.5}},
                             text=[short_name(n) if i in label_rows else "" for i, n in zip(d.index, d["committee"])],
                             textposition="middle right", cliponaxis=False,
                             customdata=d[["committee", "total", "attack_pct"]].values.tolist(),
                             hovertemplate="<b>%{customdata[0]}</b><br>Payments: %{x}<br>Total spent: $%{customdata[1]:,.0f}<br>Attack share: %{customdata[2]:.1f}%<extra></extra>"))
# Most groups are in a crowded corner: few payments and little spending
crowd = dots[(dots["total"] < 5e6) & (dots["payments"] <= 45)]
fig.add_shape(type="rect", x0=-10, x1=47, y0=-3, y1=5.2, line={"color": "#888888", "dash": "dot", "width": 1.5}, fillcolor="rgba(150,150,150,0.10)")
fig.add_annotation(x=47, y=2.5, xref="x", yref="y", ax=240, ay=-14, showarrow=True, arrowhead=2, arrowcolor="#888888", align="left",
                   text=f"<b>{len(crowd)} of {len(dots)} groups</b> are in this corner:<br>under $5M and 45 payments or fewer", font={"size": 12, "color": "#555555"})
fig.update_layout(xaxis={"title": "Number of payments", "range": [-10, dots["payments"].max() * 1.06], "zeroline": False},
                  yaxis={"title": "Total spent ($ millions)", "range": [-3, dots["total_m"].max() * 1.12], "zeroline": False},
                  legend={"orientation": "h", "x": 1, "xanchor": "right", "y": 1.04, "yanchor": "bottom", "font": {"size": 14}})
finish(fig, "09_group_dots.html", "Each outside group: how many payments and how much it spent", "Outside groups only (independent expenditures)", height=700)

## Sensitivity check
**What it means:** do the headline numbers change if we leave out rows flagged `possible_repeat` (a possible repeated payment), or if we added the affiliated committees to the campaign totals?

In [26]:
# Headline numbers under three versions of the data
def headline(camp, outs):
    """The headline numbers for one version of the data."""
    brown_day, moreno_day = daily_sides(camp, outs)
    running = (brown_day - moreno_day).cumsum()
    brown_c = camp.loc[camp["who"] == "Brown", "amount"].sum()
    moreno_c = camp.loc[camp["who"] == "Moreno", "amount"].sum()
    brown_o = outs.loc[outs["who"] == "Brown", "amount"].sum()
    moreno_o = outs.loc[outs["who"] == "Moreno", "amount"].sum()
    return {"campaign_lead_m": (brown_c - moreno_c) / 1e6,
            "net_outside_help_m": (moreno_o - brown_o) / 1e6,
            "net_outside_pct_of_lead": (moreno_o - brown_o) / (brown_c - moreno_c) * 100,
            "combined_lead_year_end_m": running.iloc[-1] / 1e6,
            "lowest_point_m": running.min() / 1e6,
            "lowest_point_date": str(running.idxmin().date()),
            "days_moreno_side_ahead": int((running < 0).sum())}

scenarios = {"Base case (used in this notebook)": (campaign, out),
             "Without possible_repeat rows": (campaign[campaign["possible_repeat"] == False], out[out["possible_repeat"] == False]),
             "Affiliated committees added to campaign totals": (pd.concat([campaign, affiliated]), out)}
sens = pd.DataFrame([{"scenario": name, **headline(c, o)} for name, (c, o) in scenarios.items()])
sens["campaign_rows_removed"] = [0, int(campaign["possible_repeat"].sum()), 0]
sens["outside_rows_removed"] = [0, int(out["possible_repeat"].sum()), 0]
sens.to_csv(OUT + "sensitivity.csv", index=False)
sens.T

,0,1,2
scenario,Base case (used in this notebook),Without possible_repeat rows,Affiliated committees added to campaign totals
campaign_lead_m,71.64,71.60,70.34
net_outside_help_m,68.37,68.39,68.37
net_outside_pct_of_lead,95.43,95.52,97.19
combined_lead_year_end_m,3.27,3.21,1.98
lowest_point_m,-10.03,-9.85,-8.54
lowest_point_date,2024-10-15,2024-10-15,2024-10-15
days_moreno_side_ahead,36,36,33
campaign_rows_removed,0,10,0
outside_rows_removed,0,29,0


## Validation
Every check compares a result with a reference number, with another result, or with its saved table. All checks must pass.

In [27]:
# Checks on every result and chart
checks = []

def check(name, expected, actual, tol=0.0):
    """Compare one expected number (or text) with the number we computed."""
    ok = (expected == actual) if isinstance(expected, str) else abs(float(expected) - float(actual)) <= tol
    checks.append({"check": name, "expected": expected, "actual": actual, "pass_fail": "PASS" if ok else "FAIL"})

def check_same(name, chart_values, table_values):
    """The values drawn in a chart must equal the values saved in its .csv table."""
    a, b = np.array(list(chart_values), dtype=float), np.array(list(table_values), dtype=float)
    ok = a.shape == b.shape and np.allclose(a, b, rtol=1e-9, atol=1e-9)
    checks.append({"check": name, "expected": float(b.sum()), "actual": float(a.sum()), "pass_fail": "PASS" if ok else "FAIL"})

def saved(filename):
    return pd.read_csv(OUT + filename)

def group_total_m(start_of_name):
    return groups.loc[groups["committee"].str.startswith(start_of_name), "total"].sum() / 1e6

# --- Reference numbers ---
check("Brown campaign total ($)", 97460584.55, side.loc["Brown", "campaign"], 0.01)
check("Moreno campaign total ($)", 25820920.56, side.loc["Moreno", "campaign"], 0.01)
check("Brown campaign lead ($)", 71639663.99, brown_lead, 0.01)
check("Outside rows (count)", 1422, len(out))
check("Outside total ($)", 286669882.57, total_outside, 0.01)
check("Outside helped Brown ($)", 109152024, side.loc["Brown", "outside"], 1)
check("Outside helped Moreno ($)", 177517859, side.loc["Moreno", "outside"], 1)
check("Outside rows helping Brown", 476, int((out["who"] == "Brown").sum()))
check("Outside rows helping Moreno", 946, int((out["who"] == "Moreno").sum()))
check("Net outside help for Moreno ($M, 1 dp)", 68.4, round(net_outside_help / 1e6, 1))
check("Share helped Moreno (%, whole number)", 62, round(who_helped.loc[0, "percent"]))
check("Share helped Brown (%, whole number)", 38, round(who_helped.loc[1, "percent"]))
t = tone.set_index(["who", "type"])
check("Brown's side attack ($M, 1 dp)", 85.6, round(t.loc[("Brown", "Attack"), "dollars"] / 1e6, 1))
check("Brown's side support ($M, 1 dp)", 23.5, round(t.loc[("Brown", "Support"), "dollars"] / 1e6, 1))
check("Moreno's side attack ($M, 1 dp)", 108.6, round(t.loc[("Moreno", "Attack"), "dollars"] / 1e6, 1))
check("Moreno's side support ($M, 1 dp)", 68.9, round(t.loc[("Moreno", "Support"), "dollars"] / 1e6, 1))
check("Brown's side attack % (whole number)", 78, round(t.loc[("Brown", "Attack"), "percent"]))
check("Moreno's side attack % (whole number)", 61, round(t.loc[("Moreno", "Attack"), "percent"]))
check("Brown's side total ($M, 1 dp)", 206.6, round(side.loc["Brown", "total"] / 1e6, 1))
check("Moreno's side total ($M, 1 dp)", 203.3, round(side.loc["Moreno", "total"] / 1e6, 1))
check("Combined lead at year-end ($M, 1 dp)", 3.3, round(combined_lead / 1e6, 1))
check("Outside share of Brown's side (%, 1 dp)", 52.8, round(side.loc["Brown", "outside_share_pct"], 1))
check("Outside share of Moreno's side (%, 1 dp)", 87.3, round(side.loc["Moreno", "outside_share_pct"], 1))
for label, day, want in [("End of 2023", "2023-12-31", 5.2), ("19 Mar 2024", "2024-03-19", 17.3), ("30 Jun 2024", "2024-06-30", 32.4),
                         ("31 Aug 2024", "2024-08-31", 30.0), ("30 Sep 2024", "2024-09-30", 0.3), ("Election Day", "2024-11-05", 1.9)]:
    check(f"Combined lead {label} ($M, 1 dp)", want, round(lead_by_day[pd.Timestamp(day)] / 1e6, 1))
check("Lowest combined lead ($M, 1 dp)", -10.0, round(lead_by_day.min() / 1e6, 1))
check("Lowest point date", "2024-10-15", str(lowest_day.date()))
check("First day Moreno's side ahead", "2024-09-24", str(days_moreno_ahead.index.min().date()))
check("Groups helping Brown", 63, int(groups.loc[groups["who"] == "Brown", "committee"].nunique()))
check("Groups helping Moreno", 30, int(groups.loc[groups["who"] == "Moreno", "committee"].nunique()))
check("Top 3 groups share, Brown's side (%, whole number)", 80, round(conc.loc[(conc["who"] == "Brown") & (conc["slice"] == "Top 3 groups"), "percent"].iloc[0]))
check("Top 3 groups share, Moreno's side (%, whole number)", 80, round(conc.loc[(conc["who"] == "Moreno") & (conc["slice"] == "Top 3 groups"), "percent"].iloc[0]))
check("WinSenate ($M, 1 dp)", 79.3, round(group_total_m("WINSENATE"), 1))
check("Senate Leadership Fund ($M, 1 dp)", 70.4, round(group_total_m("SENATE LEADERSHIP FUND"), 1))
check("Defend American Jobs ($M, 1 dp)", 40.1, round(group_total_m("DEFEND AMERICAN JOBS"), 1))
check("American Crossroads ($M, 1 dp)", 31.1, round(group_total_m("AMERICAN CROSSROADS"), 1))
check("AFP Action ($M, 1 dp)", 15.2, round(group_total_m("AMERICANS FOR PROSPERITY ACTION"), 1))
check("Sep+Oct share of Moreno's-side outside money (%, whole number)", 86, round(sep_oct_share.loc[1, "sep_oct_pct"]))
check("Sep+Oct share of Brown's-side outside money (%, whole number)", 89, round(sep_oct_share.loc[0, "sep_oct_pct"]))
cl = cat_long.set_index(["who", "category"])["percent"]
for who, name, want in [("Brown", "TV, radio & other media", 63.2), ("Brown", "Digital advertising", 29.2), ("Brown", "Canvassing & field", 5.3),
                        ("Moreno", "TV, radio & other media", 75.2), ("Moreno", "Digital advertising", 14.6), ("Moreno", "Direct mail & printing", 8.2)]:
    check(f"{LABEL[who]}: {name} (%, 1 dp)", want, round(cl.loc[(who, name)], 1))

# --- Our numbers must agree with each other ---
check("Two donut slices add to total outside ($)", total_outside, who_helped["dollars"].sum(), 0.01)
for i, who in enumerate(SIDES):
    check(f"Stacked bar segments add to {LABEL[who]} total ($M)", side.loc[who, "total"] / 1e6,
          FIGS["03_campaign_plus_outside.html"].data[0].y[i] + FIGS["03_campaign_plus_outside.html"].data[1].y[i], 1e-6)
check("Last point of the lead line = combined lead ($M)", combined_lead / 1e6, FIGS["04_lead_over_time.html"].data[0].y[-1], 1e-6)
check("Lead line has one point per day (731 days)", 731, len(lead_line))
check("Monthly bars add to Brown's-side outside total ($)", side.loc["Brown", "outside"], monthly["brown_side_usd"].sum(), 0.01)
check("Monthly bars add to Moreno's-side outside total ($)", side.loc["Moreno", "outside"], monthly["moreno_side_usd"].sum(), 0.01)
for who in SIDES:
    g = groups[groups["who"] == who].sort_values("total", ascending=False)
    check(f"Top 10 + all other groups = {LABEL[who]} outside total ($)", side.loc[who, "outside"], g["total"].head(10).sum() + g["total"].iloc[10:].sum(), 0.01)
    c = conc[conc["who"] == who]
    check(f"Top 3 + other groups donut = {LABEL[who]} outside total ($)", side.loc[who, "outside"], c["dollars"].sum(), 0.01)
    check(f"Support + attack = {LABEL[who]} outside total ($)", side.loc[who, "outside"], tone.loc[tone["who"] == who, "dollars"].sum(), 0.01)
    cc = cat_long[cat_long["who"] == who]
    check(f"Category bar adds to 100% ({LABEL[who]})", 100, cc["percent"].sum(), 1e-6)
    check(f"Category dollars = {LABEL[who]} outside total ($)", side.loc[who, "outside"], cc["dollars"].sum(), 0.01)
    d = dots[dots["who"] == who]
    check(f"Group dots add to {LABEL[who]} outside total ($)", side.loc[who, "outside"], d["total"].sum(), 0.01)
    check(f"Group dot payments = {LABEL[who]} outside rows", int((out["who"] == who).sum()), int(d["payments"].sum()))

# --- Data quality ---
check("Missing dates (campaign + outside)", 0, int(campaign["date"].isna().sum() + out["date"].isna().sum()))
check("Missing categories (campaign + outside)", 0, int(campaign["category"].isna().sum() + out["category"].isna().sum()))
check("Missing side / tone in outside file", 0, int(out["helps"].isna().sum() + out["tone"].isna().sum()))

# --- Chart files exist, and charts match their saved tables ---
check("Number of chart files saved", 10, len(CHARTS))
for name in CHARTS:
    check(f"Chart file exists: {name}", 1, int(os.path.exists(OUT + name)))

sc = saved("01_scorecard.csv")
check_same("Scorecard cards = saved table", [tr.value for tr in FIGS["01_scorecard.html"].data], sc.loc[sc["shown_as"] == "card", "value"])
f2 = FIGS["02_who_helped_donut.html"]
check_same("Who-helped donut = saved table", f2.data[0].values, saved("02_who_helped.csv")["dollars"])
f3, s3 = FIGS["03_campaign_plus_outside.html"], saved("03_campaign_plus_outside.csv")
check_same("Stacked bars (campaign) = saved table", f3.data[0].y, s3["campaign_m"])
check_same("Stacked bars (outside) = saved table", f3.data[1].y, s3["outside_m"])
check_same("Stacked bars (totals) = saved table", f3.data[2].y, s3["total_m"])
check_same("Lead line = saved table", FIGS["04_lead_over_time.html"].data[0].y, saved("04_lead_over_time.csv")["combined_lead_m"])
f5, s5 = FIGS["05_outside_by_month.html"], saved("05_outside_by_month.csv")
check_same("Monthly bars (Brown's side) = saved table", f5.data[0].y, s5.loc[s5["shown_in_chart"], "brown_side_m"])
check_same("Monthly bars (Moreno's side) = saved table", f5.data[1].y, s5.loc[s5["shown_in_chart"], "moreno_side_m"])
f6, s6 = FIGS["06a_top_groups.html"], saved("06a_top_groups.csv")
check_same("Top groups (Brown's side) = saved table", f6.data[0].x, s6.loc[s6["who"] == "Brown", "total_m"])
check_same("Top groups (Moreno's side) = saved table", f6.data[1].x, s6.loc[s6["who"] == "Moreno", "total_m"])
f6b, s6b = FIGS["06b_concentration_donuts.html"], saved("06b_concentration.csv")
check_same("Concentration donut (Brown's side) = saved table", f6b.data[0].values, s6b.loc[s6b["who"] == "Brown", "dollars"])
check_same("Concentration donut (Moreno's side) = saved table", f6b.data[1].values, s6b.loc[s6b["who"] == "Moreno", "dollars"])
f7, s7 = FIGS["07_support_vs_attack.html"], saved("07_support_vs_attack.csv")
check_same("Support/attack donut (Brown's side) = saved table", f7.data[0].values, s7.loc[s7["who"] == "Brown", "dollars"])
check_same("Support/attack donut (Moreno's side) = saved table", f7.data[1].values, s7.loc[s7["who"] == "Moreno", "dollars"])
f8, s8 = FIGS["08_communication_types.html"], saved("08_communication_types.csv")
check_same("Category bars = saved table", [v for tr in f8.data for v in tr.x], [v for name in cat_show.index for v in s8.loc[s8["category"] == name, "percent"]])
f9, s9 = FIGS["09_group_dots.html"], saved("09_group_dots.csv")
for i, who in enumerate(SIDES):
    check_same(f"Dots x = payments ({LABEL[who]}) = saved table", f9.data[i].x, s9.loc[s9["who"] == who, "payments"])
    check_same(f"Dots y = total $M ({LABEL[who]}) = saved table", f9.data[i].y, s9.loc[s9["who"] == who, "total_m"])

validation = pd.DataFrame(checks)
validation.to_csv(OUT + "validation_checks.csv", index=False)
print(len(validation), "checks;", (validation["pass_fail"] == "PASS").sum(), "pass;", (validation["pass_fail"] == "FAIL").sum(), "fail")
validation

103 checks; 103 pass; 0 fail


,check,expected,actual,pass_fail
0,Brown campaign total ($),"97,460,584.55","97,460,584.55",PASS
1,Moreno campaign total ($),"25,820,920.56","25,820,920.56",PASS
2,Brown campaign lead ($),"71,639,663.99","71,639,663.99",PASS
3,Outside rows (count),1422,1422,PASS
4,Outside total ($),"286,669,882.57","286,669,882.57",PASS
...,...,...,...,...
98,Category bars = saved table,200.00,200.00,PASS
99,Dots x = payments (Brown's side) = saved table,476.00,476.00,PASS
100,Dots y = total $M (Brown's side) = saved table,109.15,109.15,PASS
101,Dots x = payments (Moreno's side) = saved table,946.00,946.00,PASS
